In [1]:
from dotenv import load_dotenv
from playwright.async_api import expect
load_dotenv()
import os
import re

### Setup

Create empty files to save .html

In [2]:
# FOR GENERAL STORES

os.makedirs('data/general_stores_data/pages', exist_ok=True)
os.makedirs('data/general_stores_data/stores', exist_ok=True)


os.makedirs('data/country_stores_data/pages', exist_ok=True)
os.makedirs('data/country_stores_data/stores', exist_ok=True)


# FOR CHAIN STORES

os.makedirs('data/walmart_data/pages', exist_ok=True)
os.makedirs('data/walmart_data/stores', exist_ok=True)

# os.makedirs('data/dollar_general_data/pages', exist_ok=True)
# os.makedirs('data/dollar_general_data/stores', exist_ok=True)

# os.makedirs('data/dollar_tree_data/pages', exist_ok=True)
# os.makedirs('dollar_tree_data/stores', exist_ok=True)

# os.makedirs('data/family_dollar_data/pages', exist_ok=True)
# os.makedirs('data/family_dollar_data/stores', exist_ok=True)


Setup NoPecha

In [3]:
# This file is to set up my nopecha

import requests
import zipfile
import json
from utils import open_browser

# https://developers.nopecha.com/guides/extension/#loading-the-nopecha-extension-in-a-browser
with open('chromium_automation.zip', 'wb') as f:
    f.write(requests.get('https://github.com/NopeCHALLC/nopecha-extension/releases/latest/download/chromium_automation.zip').content)

with zipfile.ZipFile('chromium_automation.zip', 'r') as zip_ref:
    zip_ref.extractall("nopecha")

# Open existing manifest
with open("nopecha/manifest.json") as fp:
    data = json.load(fp)

# Update with API key
api_key = os.getenv("nopecha_APIkey")
data['nopecha']['key'] = api_key

# Save to manifest
with open("nopecha/manifest.json", "w") as fp:
    json.dump(data, fp)


### OPEN BROWSER

Define a function that opens playwright and opens a browser

In [4]:
from playwright.async_api import async_playwright, Playwright, expect, Keyboard
user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36'

async def open_browser(headless=False, user_agent=user_agent):
    """
    Starts the automated browser and opens a new window,
    in a way friendly to nopecha.

    """
    # Start playwright
    playwright = await async_playwright().start()

    # Open firefox browser, can use chromium (chrome) or others

    user_data_dir = "./user-data"
    browser = await playwright.chromium.launch_persistent_context(
        user_data_dir,
        headless=headless,
        ignore_default_args=['--enable-automation'],
        args=[
            f"--disable-extensions-except=./nopecha",
            f"--load-extension=./nopecha",
        ],
    )

    # Get an existing page, or open a new one if none exist
    page = browser.pages[0] if browser.pages else await browser.new_page()

    return browser, page

Now open the browser

In [13]:
browser, page = await open_browser()

In [ ]:
# # TESTING NOPECHA

# await page.goto("https://nopecha.com/demo")

# # Then we click the Easy reCAPTCHA link, and it loads the page. 

# await page.get_by_text("reCAPTCHA v2 (Easy").click()

Go to relevent VT biz filings webpage


In [14]:
url = 'https://bizfilings.vermont.gov/business/businesssearch'

await page.goto(url)

<Response url='https://bizfilings.vermont.gov/business/businesssearch' request=<Request url='https://bizfilings.vermont.gov/business/businesssearch' method='GET'>>

### Define general search function  (pieces of search and save)

Define a search function searches for a specific search term

In [7]:
async def store_search(page, search_term):
    '''
    Searches for stores via search term passed in as a parameter
    Specifically, types the search term into search bar and then hits search.
    '''

    # Set global action timeout to 10 seconds for all pages in this context
    page.set_default_timeout(900000)
    
    # click on the searchbar
    xpath_business_search = '//div[contains(@class, "mat-mdc-form-field-infix")]/input[@placeholder="Enter Business Name/Record Number"]'
    filter = page.locator(xpath_business_search)

    await filter.click()

    # type in search term

    await page.locator(xpath_business_search).fill(search_term)

    # click on "search" to run it

    xpath_search_button = '//span[contains (@class, "mdc-button__label") and contains(text(), "Search")]'
    search = page.locator(xpath_search_button)

    await search.click()

    # after recaptcha is compleated by nopecha (that part is automatic after setup above) click submit
    xpath_submit_recaptcha = '//span[contains(@class, "mdc-button__label") and contains(text(), "Submit")]'
    filter = page.locator(xpath_submit_recaptcha)
    await filter.click()

Define a function that saves stories

In [8]:
async def save_stores(page, page_number, filepath_forstore, next_button_xpath, search_term):

    '''
    Save .html for entire page
    Then, for each row, click on and save store-specific page.
    Then run the search again and navegate back to the page that we were on.
    '''

    # Ensure table container is there to ensure data/rows are present before continuing
    table = page.locator('tbody.mdc-data-table__content')
    await expect(table).to_be_visible()

    # SAVE .html FOR ENTIRE PAGE

    # find item range for filename
    items_forfilename = await page.locator('xpath=.//div[@class="mat-mdc-paginator-range-actions"]/div[@class="mat-mdc-paginator-range-label"]').inner_text()
    items_formatted = f"{items_forfilename}".replace(" ", "_").lower()

    # save .html
    source = await page.content()
    with open(f'{filepath_forstore}/pages/{items_formatted}.html', 'w') as f:
        f.write(source)

    # COUNT THE ROWS ON THE PAGE 
    rows = page.locator('//tbody[@class="mdc-data-table__content"]/tr')
    
    # wait for the rows to load before counting the rows
    await expect(rows.first).to_be_visible()

    # count the rows
    row_count = await rows.count()
    print(f'Row count is {row_count}.')
    
    # NOW RUN THROUGH EACH ROW AND SAVE .HTML FOR EACH ROW (STORE)
    
    for i in range(row_count):

        # identify the row
        row = rows.nth(i)

        # click on the name of the store
        await row.locator('a').click()

        # wait until page loads
        spinner = page.locator('xpath=//mat-spinner[@role="progressbar"]')
        try:
            await spinner.first.wait_for(state="visible", timeout=30000)
        except:
            pass
        await spinner.first.wait_for(state="hidden", timeout=30000)

        # after clicking into it, save the store-specific .html

        # identify store name
        store_name = [] # clearing it -- it appeared to be caching the prior name
        store_name = await page.locator('xpath=.//div[@class="row gx-3"]/div[@class="col-12 col-sm-6 col-lg-4 col-xl-3 readonly"][1]').inner_text()
        store_name_formatted = [] # clearnig
        store_name_formatted = f"{store_name}".replace("Business Name", "").replace('/', ' AKA ').strip().replace(" ", "_").title() # replacing slashes with AKA to avoid breaking when the title is passed into the file name

        # identify id number

        record_number = [] # clearing
        record_number = await page.locator('xpath=.//div[@class="row gx-3"]//div[@class="col-12 col-sm-6 col-lg-4 col-xl-3 readonly"][3]').inner_text()
        record_number_formatted = record_number.replace("Record Number", "").strip()

        source = await page.content()
        with open(f'{filepath_forstore}/stores/{store_name_formatted}_{record_number_formatted}_page-{page_number}.html', 'w') as f:
            f.write(source)

        # hit the back button
        await page.locator('xpath=//span[@class="mdc-button__label" and normalize-space()="Back"]').click()

        # redo store search
        await store_search(page, search_term)

        # navigate back to the page we were on, by clicking on next page i - 1 times
        # and click on the next button page that many times 
        for n in range(page_number - 1):
            next_button = page.locator(next_button_xpath)
            print(page_number)
            await next_button.click()
            print('clicked next')

### SEARCHES

### General Stores search

Run through each page and save general stores from each page


In [ ]:
collect = True

page_number = 1 # starting page
filepath_forstore = 'data/general_stores_data'
search_term = 'General Store'

# RUN INITIAL SEARCH 

await store_search(page, search_term)

# Define next button
# next button on pages with clickable next button 
next_button_xpath = 'xpath=.//*[contains(@class, "mat-mdc-paginator-navigation-next")]'

# next button on last page
xpath_lastpage = 'xpath=.//button[@aria-label="Next page" and contains(@class, "mat-mdc-tooltip-disabled")]'

# navigate to current page

for n in range(page_number - 1):
    next_button = page.locator(next_button_xpath)
    await next_button.click()

while collect:

    # navigate to the page we we're on
    # and click on the next button page that many times 

    for n in range(page_number - 1):
        next_button = page.locator(next_button_xpath)
        print(page_number)
        await next_button.click()

    # count rows on the page, and then use function created above to save .html for whole page as well as .html for entire row
    await save_stores(page, page_number, filepath_forstore, next_button_xpath, search_term)

    # if there's no valid next button, break this loop
    if await page.locator(xpath_lastpage).is_visible(timeout=100000):
        print("Reached the last page.")
        break

    # if there is a valid next button, go to the next page

    await page.locator(next_button_xpath).wait_for(state="visible")
    await page.locator(next_button_xpath).click(force=True)

    # wait until page loads
    
    spinner = page.locator('xpath=//mat-spinner[@role="progressbar"]')
    try:
        await spinner.first.wait_for(state="visible", timeout=30000)
    except:
        pass
    await spinner.first.wait_for(state="hidden", timeout=30000)
    
    page_number += 1

### Walmart Search

In [ ]:
collect = True

page_number = 2 # starting page
filepath_forstore = 'data/walmart_data'
search_term = 'Walmart'

# RUN INITIAL SEARCH 

await store_search(page, search_term)

# Define next button
# next button on pages with clickable next button 
next_button_xpath = 'xpath=.//*[contains(@class, "mat-mdc-paginator-navigation-next")]'

# next button on last page
xpath_lastpage = 'xpath=.//button[@aria-label="Next page" and contains(@class, "mat-mdc-tooltip-disabled")]'

# navigate to the page we we're on
for n in range(page_number - 1):
    next_button = page.locator(next_button_xpath)
    await next_button.click()

while collect:

    # count rows on the page, and then use function created above to save .html for whole page as well as .html for entire row
    await save_stores(page, page_number, filepath_forstore, next_button_xpath, search_term)

    # if there's no valid next button, break this loop
    if await page.locator(xpath_lastpage).is_visible(timeout=100000):
        print("Reached the last page.")
        break

    # if there is a valid next button, go to the next page

    await page.locator(next_button_xpath).wait_for(state="visible")
    await page.locator(next_button_xpath).click(force=True)

    # wait until page loads
    
    spinner = page.locator('xpath=//mat-spinner[@role="progressbar"]')
    try:
        await spinner.first.wait_for(state="visible", timeout=30000)
    except:
        pass
    await spinner.first.wait_for(state="hidden", timeout=30000)
    
    page_number += 1

### 'County Store' Search

In [15]:
collect = True

page_number = 12 # starting page
filepath_forstore = 'data/country_stores_data'
search_term = 'Country Store'

# RUN INITIAL SEARCH 

await store_search(page, search_term)

# Define next button
# next button on pages with clickable next button 
next_button_xpath = 'xpath=.//*[contains(@class, "mat-mdc-paginator-navigation-next")]'

# next button on last page
xpath_lastpage = 'xpath=.//button[@aria-label="Next page" and contains(@class, "mat-mdc-tooltip-disabled")]'

# navigate to the page we we're on
# and click on the next button page that many times 

for n in range(page_number - 1):
    next_button = page.locator(next_button_xpath)
    await next_button.click()

while collect:

    # count rows on the page, and then use function created above to save .html for whole page as well as .html for entire row
    await save_stores(page, page_number, filepath_forstore, next_button_xpath, search_term)

    # if there's no valid next button, break this loop
    if await page.locator(xpath_lastpage).is_visible(timeout=100000):
        print("Reached the last page.")
        break

    # if there is a valid next button, go to the next page

    await page.locator(next_button_xpath).wait_for(state="visible")
    await page.locator(next_button_xpath).click(force=True)

    # wait until page loads
    
    spinner = page.locator('xpath=//mat-spinner[@role="progressbar"]')
    try:
        await spinner.first.wait_for(state="visible", timeout=30000)
    except:
        pass
    await spinner.first.wait_for(state="hidden", timeout=30000)
    
    page_number += 1


Row count is 25.
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clicked next
12
clic

TimeoutError: Locator.click: Timeout 900000ms exceeded.
Call log:
  - waiting for locator("//span[contains(@class, \"mdc-button__label\") and contains(text(), \"Submit\")]")
    - locator resolved to <span class="mdc-button__label"> Submit </span>
  - attempting click action
    2 × waiting for element to be visible, enabled and stable
      - element is not stable
    - retrying click action
    - waiting 20ms
    - waiting for element to be visible, enabled and stable
    - element is not stable
  2 × retrying click action
      - waiting 100ms
      - waiting for element to be visible, enabled and stable
      - element is not enabled
  1740 × retrying click action
         - waiting 500ms
         - waiting for element to be visible, enabled and stable
         - element is not enabled
  - retrying click action
    - waiting 500ms


FOR GENERAL STORES, ENSURE NO DUPLICATES OR MISSING FILES

Run this to ensure that it downloaded each store record, and to check for duplicates

In [ ]:
import re
import pandas as pd

# CREATE LIST OF ALL RECORD NUMBERS OF ALREADY DOWNLOADED STORES
all_files = []
record_numbers = []

for filename in os.listdir('data/general_stores_data/stores/'):
    all_files.append(filename)
    match = re.search(r'_(\d+)_page-\d+\.html$', filename)
    if match:
        record_numbers.append(match.group(1))
    else:
        print(f'{filename} has a record error.')

print(len(all_files)) # just for debugging
print(record_numbers)

# print('---')

# # check for duplicates (just to note; leaving in for now and will deal with it in the cleaning file)
# duplicates = pd.Series(record_numbers)[pd.Series(record_numbers).duplicated()].unique()
# print(duplicates)

Run through all rows on all pages, read record number, and make sure that record number exists in the saved data folder

In [ ]:
# DEFINE UNIVERSAL VARIABLES

search_term = 'General Store'

# next button on pages with clickable next button 
next_button_xpath = 'xpath=.//*[contains(@class, "mat-mdc-paginator-navigation-next")]'

# next button on last page
xpath_lastpage = 'xpath=.//button[@aria-label="Next page" and contains(@class, "mat-mdc-tooltip-disabled")]'

search_term = 'General Store'

url = 'https://bizfilings.vermont.gov/business/businesssearch'

# OPEN BROWSER

browser, page = await open_browser()

# RUN SEARCH 

await page.goto(url)
await store_search(page, search_term)

RUNNING THIS LOOP TWICE: \
FIRST TO DIAGNOSE

In [ ]:
collect = True
page_number = 1 # starting page

already_recorded = []
not_yet_recorded = []

while collect:

    # LOAD PAGE

    # Ensure table container is there to ensure data/rows are present before continuing
    table = page.locator('tbody.mdc-data-table__content')
    await expect(table).to_be_visible()
    
    # COUNT ROWS SO I CAN THEN RUN THROUGH THEM 

    rows = page.locator('//tbody[@class="mdc-data-table__content"]/tr')
    await expect(rows.first).to_be_visible() # wait for rows to load
    row_count = await rows.count()

    # now run through each row
    # identify the record number on the page
    for i in range(row_count):

        # identify the row
        row = rows.nth(i)
        row_record_number = await row.locator('xpath=//td[@role="cell"][2]').inner_text()

        # look for it in the record_numbers list of already saved store .htmls
        # if already saved, do nothing
        if row_record_number in record_numbers:
            already_recorded.append(row_record_number)
            pass 
        # if not yet saved, print it out
        elif row_record_number not in record_numbers:
            print(f'{row_record_number} not yet recorded.')
            not_yet_recorded.append(row_record_number)

            # THEN SAVE IT 

            # identify the row
    
            # click on the name of the store
            await row.locator('a').click()
    
            # wait until page loads
            spinner = page.locator('xpath=//mat-spinner[@role="progressbar"]')
            try:
                await spinner.first.wait_for(state="visible", timeout=30000)
            except:
                pass
            await spinner.first.wait_for(state="hidden", timeout=30000)
    
            # after clicking into it, save the store-specific .html
    
            # identify store name
            store_name = [] # clearing it -- it appeared to be caching the prior name
            store_name = await page.locator('xpath=.//div[@class="row gx-3"]/div[@class="col-12 col-sm-6 col-lg-4 col-xl-3 readonly"][1]').inner_text()
            store_name_formatted = [] # clearnig
            store_name_formatted = f"{store_name}".replace("Business Name", "").strip().replace(" ", "_").title()
    
            # identify id number
    
            record_number = [] # clearing
            record_number = await page.locator('xpath=.//div[@class="row gx-3"]//div[@class="col-12 col-sm-6 col-lg-4 col-xl-3 readonly"][3]').inner_text()
            record_number_formatted = record_number.replace("Record Number", "").strip()
    
            source = await page.content()
            with open(f'data/general_stores_data/stores/{store_name_formatted}_{record_number_formatted}_page-{page_number}.html', 'w') as f:
                f.write(source)
    
            # hit the back button
            await page.locator('xpath=//span[@class="mdc-button__label" and normalize-space()="Back"]').click()
    
            # redo store search
            await store_search(page, search_term)
    
            # navigate back to the page we were on, by clicking on next page i - 1 times
            # and click on the next button page that many times
            
            for n in range(page_number - 1):
                next_button = page.locator(next_button_xpath)
                await next_button.click()            

    # IF NEXT PAGE, MOVE TO NEXT PAGE 
    # if there's no valid next button, break this loop
    if await page.locator(xpath_lastpage).is_visible(timeout=100000):
        print("Reached the last page.")
        break

    # if there is a valid next button, go to the next page

    await page.locator(next_button_xpath).wait_for(state="visible")
    await page.locator(next_button_xpath).click(force=True)

    # wait until page loads
    
    spinner = page.locator('xpath=//mat-spinner[@role="progressbar"]')
    try:
        await spinner.first.wait_for(state="visible", timeout=30000)
    except:
        pass
    await spinner.first.wait_for(state="hidden", timeout=30000)
    
    page_number += 1

print(f'{len(already_recorded)} have been successfully downloaded, while {len(not_yet_recorded)} are missing from my downloads.')

noting initial results (output from above) \
\
&emsp;158865 not yet recorded. \
&emsp;183587 not yet recorded. \
&emsp;420049 not yet recorded. \
&emsp;Reached the last page. \
&emsp;468 have been successfully downloaded, while 3 are missing from my downloads. \

Now check no missing values.

First, re-make the list of existing files' record numbers so that it includes those we just downloaded.

In [ ]:
# CREATE LIST OF ALL RECORD NUMBERS OF ALREADY DOWNLOADED STORES
all_files = []
record_numbers = []

for filename in os.listdir('data/general_stores_data/stores/'):
    all_files.append(filename)
    match = re.search(r'_(\d+)_page-\d+\.html$', filename)
    if match:
        record_numbers.append(match.group(1))
    else:
        print(f'{filename} has a record error.')

print(len(all_files)) # just for debugging
print(record_numbers)


Then re-run same loop as above

In [ ]:
collect = True
page_number = 1 # starting page

already_recorded = []
not_yet_recorded = []

while collect:

    # LOAD PAGE

    # Ensure table container is there to ensure data/rows are present before continuing
    table = page.locator('tbody.mdc-data-table__content')
    await expect(table).to_be_visible()
    
    # COUNT ROWS SO I CAN THEN RUN THROUGH THEM 

    rows = page.locator('//tbody[@class="mdc-data-table__content"]/tr')
    await expect(rows.first).to_be_visible() # wait for rows to load
    row_count = await rows.count()

    # now run through each row
    # identify the record number on the page
    for i in range(row_count):

        # identify the row
        row = rows.nth(i)
        row_record_number = await row.locator('xpath=//td[@role="cell"][2]').inner_text()

        # look for it in the record_numbers list of already saved store .htmls
        # if already saved, do nothing
        if row_record_number in record_numbers:
            already_recorded.append(row_record_number)
            pass 
        # if not yet saved, print it out
        elif row_record_number not in record_numbers:
            print(f'{row_record_number} not yet recorded.')
            not_yet_recorded.append(row_record_number)

            # THEN SAVE IT 

            # identify the row
    
            # click on the name of the store
            await row.locator('a').click()
    
            # wait until page loads
            spinner = page.locator('xpath=//mat-spinner[@role="progressbar"]')
            try:
                await spinner.first.wait_for(state="visible", timeout=30000)
            except:
                pass
            await spinner.first.wait_for(state="hidden", timeout=30000)
    
            # after clicking into it, save the store-specific .html
    
            # identify store name
            store_name = [] # clearing it -- it appeared to be caching the prior name
            store_name = await page.locator('xpath=.//div[@class="row gx-3"]/div[@class="col-12 col-sm-6 col-lg-4 col-xl-3 readonly"][1]').inner_text()
            store_name_formatted = [] # clearnig
            store_name_formatted = f"{store_name}".replace("Business Name", "").strip().replace(" ", "_").title()
    
            # identify id number
    
            record_number = [] # clearing
            record_number = await page.locator('xpath=.//div[@class="row gx-3"]//div[@class="col-12 col-sm-6 col-lg-4 col-xl-3 readonly"][3]').inner_text()
            record_number_formatted = record_number.replace("Record Number", "").strip()
    
            source = await page.content()
            with open(f'data/general_stores_data/stores/{store_name_formatted}_{record_number_formatted}_page-{page_number}.html', 'w') as f:
                f.write(source)
    
            # hit the back button
            await page.locator('xpath=//span[@class="mdc-button__label" and normalize-space()="Back"]').click()
    
            # redo store search
            await store_search(page, search_term)
    
            # navigate back to the page we were on, by clicking on next page i - 1 times
            # and click on the next button page that many times
            
            for n in range(page_number - 1):
                next_button = page.locator(next_button_xpath)
                print(page_number)
                await next_button.click()
                print('clicked next')
            

    # IF NEXT PAGE, MOVE TO NEXT PAGE 
    # if there's no valid next button, break this loop
    if await page.locator(xpath_lastpage).is_visible(timeout=100000):
        print("Reached the last page.")
        break

    # if there is a valid next button, go to the next page

    await page.locator(next_button_xpath).wait_for(state="visible")
    await page.locator(next_button_xpath).click(force=True)

    # wait until page loads
    
    spinner = page.locator('xpath=//mat-spinner[@role="progressbar"]')
    try:
        await spinner.first.wait_for(state="visible", timeout=30000)
    except:
        pass
    await spinner.first.wait_for(state="hidden", timeout=30000)
    
    page_number += 1

    # # if there is a valid next button, create a new tab and go to the next page
    # new_tab = await browser.new_page() # creating a new tab to mitigate some loading/caching errors I was having

    # await new_tab.locator(next_button_xpath).wait_for(state="visible")

    # page = new_tab
    
    # await new_tab.locator(next_button_xpath).click(force=True)

print(f'{len(already_recorded)} have been successfully downloaded, while {len(not_yet_recorded)} are missing from my downloads.')